# EE/CS 148B HW3: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime.
- We recommend using an A100 runtime.
- Put the `hw3` directory somewhere accessible from Colab, typically Google Drive.

This notebook assumes the repo already exists and only sets up Python dependencies plus the repo import path.

Note: We will be using `pip` for dependencies inside Colab.

In [1]:
%%capture
!pip -q install -U torch torchvision "transformers>=4.45.0,<4.57.0" datasets "sentence-transformers<4.0" accelerate pillow tqdm matplotlib wandb pyyaml einops pytest pytest-cov
!pip -q install ninja packaging
!pip -q install flash-attn --no-build-isolation

In [2]:
!pip install -q --force-reinstall "transformers>=4.45.0,<4.57.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is inco

In [3]:
# NumPy version check — Colab's pre-installed packages require NumPy 2.x;
# do NOT downgrade to 1.x here.
import numpy as np; print("numpy", np.__version__)

numpy 2.4.6


In [16]:
!git clone https://github.com/wduan10/cs148b_hw3.git /content/cs148b_hw3
%cd /content/cs148b_hw3

Cloning into '/content/cs148b_hw3'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 43 (delta 0), reused 43 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 426.81 KiB | 14.23 MiB/s, done.
/content/cs148b_hw3


In [18]:
from pathlib import Path

USE_DRIVE = False
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/hw3/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/cs148b_hw3')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


Using repo: /content/cs148b_hw3


In [ ]:
# Pull latest changes (re-run this whenever you push updates from local)
!git -C /content/cs148b_hw3 pull

In [19]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

cwd = /content/cs148b_hw3


In [20]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

torch 2.12.0+cu130 | transformers 4.56.2


In [23]:
import gc
import math
import random
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

import basics
import vlm
from basics.lora import LoRALinear, apply_lora_to_attention
from basics.model import Block, MultiHeadAttention
from basics.rope import RoPE1D, RoPE2D
from basics.text_encoder import FrozenTextEncoder
from basics.vit import PatchEmbeddings, ViT
from vlm.clip import ProjectionHeads, clip_loss, init_logit_scale
from vlm.data import (
    EUROSAT_CLASSES,
    build_clevr_loaders,
    build_eurosat_loaders,
    build_resisc45_loaders,
)
from vlm.eval import batch_clevr_accuracy, clevr_exact_match, zeroshot_classification_accuracy
from vlm.masking import build_image_bidir_mask
from vlm.model import VisionLanguageModel
from vlm.projector import VisionLanguageProjector

SEED = 0
set_seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass

CONFIGS = {
    'clip': REPO_ROOT / 'configs' / 'clip_eurosat.yaml',
    'lora': REPO_ROOT / 'configs' / 'lora_resisc.yaml',
    'vlm': REPO_ROOT / 'configs' / 'vlm_clevr.yaml',
}
for name, path in CONFIGS.items():
    assert path.exists(), f'Missing {name} config: {path}'
    with open(path) as f:
        _ = yaml.safe_load(f)

print('HW3 imports and configs loaded.')


ModuleNotFoundError: No module named 'basics.lora'

## Your HW3 code starts here